# 03 - Task 2 models (coarse-to-fine with weighted ensemble)

This notebook trains and compares four Task 2 models using **5-fold stratified CV** on the full Task 2 training set:

1. Calibrated Linear SVM on handcrafted features
2. RBF SVM on handcrafted features (small `C`/`gamma` grid)
3. Logistic Regression on ResNet18 embeddings
4. Weighted soft-voting ensemble over models 1-3

Outputs written to `outputs/`:

- experiment rows in `outputs/metrics/log.csv`
- confusion matrices in `outputs/figures/task2/`
- predictions in `outputs/predictions/`


In [1]:
import json
import shutil
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC, SVC

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from utils import (
    evaluate,
    kfold,
    load_embedding_cache,
    load_task,
    log_experiment,
    make_submission,
    plot_confusion,
    seed_everything,
)

SEED = 42
seed_everything(SEED)

OUT_PRED = REPO_ROOT / "outputs" / "predictions"
OUT_FIG = REPO_ROOT / "outputs" / "figures" / "task2"
OUT_PRED.mkdir(parents=True, exist_ok=True)
OUT_FIG.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_colwidth", 80)


In [2]:
bundle = load_task(2)

X_hand = bundle.X_handcrafted_train
X_hand_test = bundle.X_handcrafted_test
y = bundle.y_train
train_ids = bundle.train_ids
test_ids = bundle.test_ids
class_mapping = bundle.class_mapping.sort_values("class_id").reset_index(drop=True)
class_ids = class_mapping["class_id"].tolist()
class_names = (
    class_mapping["class_name"].tolist()
    if "class_name" in class_mapping.columns
    else [str(cid) for cid in class_ids]
)

X_resnet, resnet_train_ids = load_embedding_cache(2, "train", "resnet18")
X_resnet_test, resnet_test_ids = load_embedding_cache(2, "test", "resnet18")

assert list(resnet_train_ids) == list(train_ids)
assert list(resnet_test_ids) == list(test_ids)

print(f"Task 2 train size: {len(y)}")
print(f"Task 2 test size: {len(test_ids)}")
print(f"Handcrafted dim: {X_hand.shape[1]} | ResNet18 dim: {X_resnet.shape[1]}")


Task 2 train size: 417
Task 2 test size: 180
Handcrafted dim: 219 | ResNet18 dim: 512


In [3]:
def make_calibrated_linear_svm(seed: int = SEED):
    base = LinearSVC(C=1.0, random_state=seed, max_iter=5000)
    try:
        calibrated = CalibratedClassifierCV(estimator=base, method="sigmoid", cv=3)
    except TypeError:
        calibrated = CalibratedClassifierCV(base_estimator=base, method="sigmoid", cv=3)
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            ("clf", calibrated),
        ]
    )


def make_rbf_svm(c: float, gamma, seed: int = SEED):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "clf",
                SVC(
                    C=c,
                    gamma=gamma,
                    kernel="rbf",
                    probability=True,
                    random_state=seed,
                ),
            ),
        ]
    )


def make_lr_resnet(seed: int = SEED):
    return Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "clf",
                LogisticRegression(
                    max_iter=3000,
                    C=1.0,
                    solver="lbfgs",
                    random_state=seed,
                ),
            ),
        ]
    )


rbf_grid = [
    {"C": 1.0, "gamma": "scale"},
    {"C": 3.0, "gamma": "scale"},
    {"C": 10.0, "gamma": "scale"},
    {"C": 10.0, "gamma": 0.01},
    {"C": 30.0, "gamma": 0.005},
]

rbf_rows = []
for params in rbf_grid:
    est = make_rbf_svm(params["C"], params["gamma"])
    cv = cross_validate(
        est,
        X_hand,
        y,
        cv=kfold(5, seed=SEED),
        scoring={"acc": "accuracy", "macro_f1": "f1_macro"},
        return_train_score=False,
        n_jobs=-1,
    )
    rbf_rows.append(
        {
            "C": params["C"],
            "gamma": params["gamma"],
            "cv_acc_mean": float(np.mean(cv["test_acc"])),
            "cv_macro_f1_mean": float(np.mean(cv["test_macro_f1"])),
        }
    )

rbf_search_df = pd.DataFrame(rbf_rows).sort_values(
    by=["cv_macro_f1_mean", "cv_acc_mean"],
    ascending=False,
).reset_index(drop=True)
best_rbf = rbf_search_df.iloc[0].to_dict()
best_rbf_c = float(best_rbf["C"])
best_rbf_gamma = best_rbf["gamma"]

print("RBF SVM grid search (Task 2, 5-fold CV):")
rbf_search_df


RBF SVM grid search (Task 2, 5-fold CV):


,C,gamma,cv_acc_mean,cv_macro_f1_mean
0,10.0,scale,0.246845,0.237394
1,3.0,scale,0.246873,0.236937
2,30.0,0.005,0.237263,0.226016
3,10.0,0.01,0.225559,0.210638
4,1.0,scale,0.218302,0.184535


In [4]:
model_specs = {
    "calibrated_linear_svm_handcrafted": {
        "feature_set": "handcrafted_219d",
        "X_train": X_hand,
        "X_test": X_hand_test,
        "build": lambda: make_calibrated_linear_svm(SEED),
        "hyperparams": {"C": 1.0, "calibration_cv": 3, "method": "sigmoid"},
    },
    "svm_rbf_handcrafted": {
        "feature_set": "handcrafted_219d",
        "X_train": X_hand,
        "X_test": X_hand_test,
        "build": lambda: make_rbf_svm(best_rbf_c, best_rbf_gamma, SEED),
        "hyperparams": {"C": best_rbf_c, "gamma": best_rbf_gamma},
    },
    "lr_resnet18": {
        "feature_set": "resnet18_512d",
        "X_train": X_resnet,
        "X_test": X_resnet_test,
        "build": lambda: make_lr_resnet(SEED),
        "hyperparams": {"C": 1.0, "solver": "lbfgs", "max_iter": 3000},
    },
}

results = []
fitted_full_models = {}

for model_name, spec in model_specs.items():
    print(f"\n=== {model_name} ===")
    est = spec["build"]()
    X_train_local = spec["X_train"]

    cv = cross_validate(
        clone(est),
        X_train_local,
        y,
        cv=kfold(5, seed=SEED),
        scoring={"acc": "accuracy", "macro_f1": "f1_macro"},
        return_train_score=False,
        n_jobs=-1,
    )

    oof_pred = cross_val_predict(
        clone(est),
        X_train_local,
        y,
        cv=kfold(5, seed=SEED),
        method="predict",
        n_jobs=-1,
    )
    metrics = evaluate(y, oof_pred, labels=class_ids)

    cm_path = OUT_FIG / f"{model_name}_cv_confusion.png"
    plot_confusion(
        metrics["confusion_matrix"],
        labels=class_names,
        out_path=cm_path,
        title=f"Task 2 CV confusion - {model_name}",
        normalize=True,
    )

    full_model = clone(est)
    t0 = time.perf_counter()
    full_model.fit(X_train_local, y)
    train_time_s = time.perf_counter() - t0
    fitted_full_models[model_name] = full_model

    y_test_pred = full_model.predict(spec["X_test"])
    out_csv = OUT_PRED / f"task2_{model_name}_class_id.csv"
    make_submission(
        test_ids=test_ids,
        y_pred=y_test_pred,
        class_mapping=class_mapping,
        out_path=out_csv,
        label_column="class_id",
    )

    log_row = {
        "task": 2,
        "model": model_name,
        "feature_set": spec["feature_set"],
        "hyperparams": json.dumps(spec["hyperparams"], default=str),
        "cv_mean_acc": float(np.mean(cv["test_acc"])),
        "cv_std_acc": float(np.std(cv["test_acc"])),
        "cv_mean_macro_f1": float(np.mean(cv["test_macro_f1"])),
        "cv_std_macro_f1": float(np.std(cv["test_macro_f1"])),
        "val_acc": float(metrics["accuracy"]),
        "val_macro_f1": float(metrics["macro_f1"]),
        "train_time_s": float(train_time_s),
        "notes": "task2_cv5_fulltrain",
    }
    log_experiment(log_row)

    results.append(
        {
            "model": model_name,
            "feature_set": spec["feature_set"],
            "cv_acc_mean": log_row["cv_mean_acc"],
            "cv_acc_std": log_row["cv_std_acc"],
            "cv_macro_f1_mean": log_row["cv_mean_macro_f1"],
            "cv_macro_f1_std": log_row["cv_std_macro_f1"],
            "oof_acc": log_row["val_acc"],
            "oof_macro_f1": log_row["val_macro_f1"],
            "full_fit_time_s": log_row["train_time_s"],
            "confusion_png": str(cm_path.relative_to(REPO_ROOT)),
            "submission_csv": str(out_csv.relative_to(REPO_ROOT)),
        }
    )

    print(f"CV acc={log_row['cv_mean_acc']:.4f} +/- {log_row['cv_std_acc']:.4f}")
    print(f"CV f1 ={log_row['cv_mean_macro_f1']:.4f} +/- {log_row['cv_std_macro_f1']:.4f}")
    print(f"OOF acc={log_row['val_acc']:.4f} | OOF macro-F1={log_row['val_macro_f1']:.4f}")
    print(f"Saved confusion matrix: {cm_path}")
    print(f"Saved submission CSV: {out_csv}")



=== calibrated_linear_svm_handcrafted ===


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runtim

CV acc=0.2303 +/- 0.0414
CV f1 =0.2108 +/- 0.0370
OOF acc=0.2302 | OOF macro-F1=0.2163
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task2/calibrated_linear_svm_handcrafted_cv_confusion.png
Saved submission CSV: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/predictions/task2_calibrated_linear_svm_handcrafted_class_id.csv

=== svm_rbf_handcrafted ===
CV acc=0.2468 +/- 0.0410
CV f1 =0.2374 +/- 0.0372
OOF acc=0.2470 | OOF macro-F1=0.2425
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task2/svm_rbf_handcrafted_cv_confusion.png
Saved submission CSV: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/predictions/task2_svm_rbf_handcrafted_class_id.csv

=== lr_resnet18 ===


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * we

CV acc=0.8394 +/- 0.0340
CV f1 =0.8388 +/- 0.0337
OOF acc=0.8393 | OOF macro-F1=0.8408
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task2/lr_resnet18_cv_confusion.png
Saved submission CSV: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/predictions/task2_lr_resnet18_class_id.csv


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights.T + intercept  # ndarray, likely C-contiguous
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:336: RuntimeWarning: divide by zero encountered in matmul
  grad[:, :n_features] = grad_pointwise.T @ X + l2_reg_strength * we

In [5]:
base_for_ensemble = [
    "calibrated_linear_svm_handcrafted",
    "svm_rbf_handcrafted",
    "lr_resnet18",
]

cv_weight_map = {}
for row in results:
    if row["model"] in base_for_ensemble:
        cv_weight_map[row["model"]] = max(row["cv_macro_f1_mean"], 1e-8)

ensemble_weights = np.array([cv_weight_map[name] for name in base_for_ensemble], dtype=np.float64)
ensemble_weights = ensemble_weights / ensemble_weights.sum()

oof_ensemble_pred = np.empty_like(y)
fold_acc = []
fold_macro_f1 = []

for train_idx, val_idx in kfold(5, seed=SEED).split(X_hand, y):
    y_train_fold = y[train_idx]

    fold_specs = [
        ("calibrated_linear_svm_handcrafted", make_calibrated_linear_svm(SEED), X_hand),
        ("svm_rbf_handcrafted", make_rbf_svm(best_rbf_c, best_rbf_gamma, SEED), X_hand),
        ("lr_resnet18", make_lr_resnet(SEED), X_resnet),
    ]

    weighted_probs = None
    ref_classes = None

    for i, (_, est, X_full) in enumerate(fold_specs):
        model_fold = clone(est)
        model_fold.fit(X_full[train_idx], y_train_fold)
        probs = model_fold.predict_proba(X_full[val_idx])
        if ref_classes is None:
            ref_classes = model_fold.classes_
            weighted_probs = np.zeros_like(probs, dtype=np.float64)
        else:
            assert np.array_equal(ref_classes, model_fold.classes_), "classes_ ordering mismatch"
        weighted_probs += ensemble_weights[i] * probs

    fold_pred = ref_classes[np.argmax(weighted_probs, axis=1)]
    oof_ensemble_pred[val_idx] = fold_pred

    y_val_fold = y[val_idx]
    fold_acc.append(float(np.mean(fold_pred == y_val_fold)))
    fold_macro_f1.append(
        float(evaluate(y_val_fold, fold_pred, labels=class_ids)["macro_f1"])
    )

ensemble_metrics = evaluate(y, oof_ensemble_pred, labels=class_ids)
ensemble_cm_path = OUT_FIG / "weighted_soft_vote_ensemble_cv_confusion.png"
plot_confusion(
    ensemble_metrics["confusion_matrix"],
    labels=class_names,
    out_path=ensemble_cm_path,
    title="Task 2 CV confusion - weighted soft-voting ensemble",
    normalize=True,
)

full_probs = []
full_ref_classes = None
for name in base_for_ensemble:
    fitted = fitted_full_models[name]
    probs = fitted.predict_proba(model_specs[name]["X_test"])
    if full_ref_classes is None:
        full_ref_classes = fitted.classes_
    else:
        assert np.array_equal(full_ref_classes, fitted.classes_), "full-model classes_ mismatch"
    full_probs.append(probs)

ensemble_test_probs = np.zeros_like(full_probs[0], dtype=np.float64)
for i, probs in enumerate(full_probs):
    ensemble_test_probs += ensemble_weights[i] * probs
ensemble_test_pred = full_ref_classes[np.argmax(ensemble_test_probs, axis=1)]

ensemble_csv = OUT_PRED / "task2_weighted_soft_vote_ensemble_class_id.csv"
make_submission(
    test_ids=test_ids,
    y_pred=ensemble_test_pred,
    class_mapping=class_mapping,
    out_path=ensemble_csv,
    label_column="class_id",
)

ensemble_row = {
    "task": 2,
    "model": "weighted_soft_vote_ensemble",
    "feature_set": "handcrafted_219d+resnet18_512d",
    "hyperparams": json.dumps(
        {
            "components": base_for_ensemble,
            "weights": {name: float(w) for name, w in zip(base_for_ensemble, ensemble_weights)},
            "rbf_c": best_rbf_c,
            "rbf_gamma": best_rbf_gamma,
        },
        default=str,
    ),
    "cv_mean_acc": float(np.mean(fold_acc)),
    "cv_std_acc": float(np.std(fold_acc)),
    "cv_mean_macro_f1": float(np.mean(fold_macro_f1)),
    "cv_std_macro_f1": float(np.std(fold_macro_f1)),
    "val_acc": float(ensemble_metrics["accuracy"]),
    "val_macro_f1": float(ensemble_metrics["macro_f1"]),
    "train_time_s": np.nan,
    "notes": "task2_cv5_weighted_soft_vote",
}
log_experiment(ensemble_row)

results.append(
    {
        "model": ensemble_row["model"],
        "feature_set": ensemble_row["feature_set"],
        "cv_acc_mean": ensemble_row["cv_mean_acc"],
        "cv_acc_std": ensemble_row["cv_std_acc"],
        "cv_macro_f1_mean": ensemble_row["cv_mean_macro_f1"],
        "cv_macro_f1_std": ensemble_row["cv_std_macro_f1"],
        "oof_acc": ensemble_row["val_acc"],
        "oof_macro_f1": ensemble_row["val_macro_f1"],
        "full_fit_time_s": np.nan,
        "confusion_png": str(ensemble_cm_path.relative_to(REPO_ROOT)),
        "submission_csv": str(ensemble_csv.relative_to(REPO_ROOT)),
    }
)

print("Ensemble weights (from base-model CV macro-F1):")
for name, w in zip(base_for_ensemble, ensemble_weights):
    print(f"  {name}: {w:.4f}")
print(f"Saved confusion matrix: {ensemble_cm_path}")
print(f"Saved submission CSV: {ensemble_csv}")


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runtim

Ensemble weights (from base-model CV macro-F1):
  calibrated_linear_svm_handcrafted: 0.1638
  svm_rbf_handcrafted: 0.1845
  lr_resnet18: 0.6518
Saved confusion matrix: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/figures/task2/weighted_soft_vote_ensemble_cv_confusion.png
Saved submission CSV: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/predictions/task2_weighted_soft_vote_ensemble_class_id.csv


/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/skandaramanan/Documents/ML/MLAssignment2/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: Runtim

In [6]:
results_df = pd.DataFrame(results).sort_values(
    by=["cv_macro_f1_mean", "cv_acc_mean"],
    ascending=False,
).reset_index(drop=True)

best_model = results_df.iloc[0]["model"]
best_src = OUT_PRED / f"task2_{best_model}_class_id.csv"
canonical_path = OUT_PRED / "task2_submission_class_id.csv"

shutil.copyfile(best_src, canonical_path)

print(f"Best CV model: {best_model}")
print(f"Canonical submission: {canonical_path}")
results_df


Best CV model: lr_resnet18
Canonical submission: /Users/skandaramanan/Documents/ML/MLAssignment2/outputs/predictions/task2_submission_class_id.csv


,model,feature_set,cv_acc_mean,cv_acc_std,cv_macro_f1_mean,cv_macro_f1_std,oof_acc,oof_macro_f1,full_fit_time_s,confusion_png,submission_csv
0,lr_resnet18,resnet18_512d,0.839415,0.034037,0.838816,0.033737,0.839329,0.840798,0.029268,outputs/figures/task2/lr_resnet18_cv_confusion.png,outputs/predictions/task2_lr_resnet18_class_id.csv
1,weighted_soft_vote_ensemble,handcrafted_219d+resnet18_512d,0.837034,0.037152,0.836616,0.036552,0.836930,0.838405,NaN,outputs/figures/task2/weighted_soft_vote_ensemble_cv_confusion.png,outputs/predictions/task2_weighted_soft_vote_ensemble_class_id.csv
2,svm_rbf_handcrafted,handcrafted_219d,0.246845,0.041048,0.237394,0.037179,0.247002,0.242463,0.165570,outputs/figures/task2/svm_rbf_handcrafted_cv_confusion.png,outputs/predictions/task2_svm_rbf_handcrafted_class_id.csv
3,calibrated_linear_svm_handcrafted,handcrafted_219d,0.230264,0.041350,0.210781,0.036953,0.230216,0.216317,5.527834,outputs/figures/task2/calibrated_linear_svm_handcrafted_cv_confusion.png,outputs/predictions/task2_calibrated_linear_svm_handcrafted_class_id.csv
